# Environment / Dependencies Setup
One command installs everything the lab needs:

In [ ]:
!pip install -q "pymongo[srv,tls]==4.10.1" python-dotenv==1.0.1 certifi

### Step 1 — Connect to MongoDB
`load_dotenv()` reads `MONGODB_URI` from `.env`, and `certifi` supplies the CA bundle for a secure TLS connection.

In [ ]:
import os
import certifi
from dotenv import load_dotenv
import pymongo

# Load the Atlas connection string from the .env file
load_dotenv()
uri = os.environ["MONGODB_URI"]

# Connect to the real MongoDB Atlas cluster
# tlsCAFile uses certifi's CA bundle so SSL works reliably on all platforms
client = pymongo.MongoClient(uri, tlsCAFile=certifi.where())

# Access (or create) the database and collection
db = client["school_db"]
students = db["students"]

print("Connected to MongoDB Atlas")

### Step 2 — Design and Insert Student Records
`students.drop()` clears the collection, then `insert_many()` adds the 25 student documents in one call.

In [ ]:
# Drop the collection first so re-running the notebook starts clean
# Without this, re-running would duplicate all 25 records
students.drop()

# 25 student records across 5 courses with varying grades
# Some students have grades below 60 (failing) to make queries interesting
student_records = [
    {"name": "Alice Johnson",    "student_id": "STU001", "course": "Computer Science", "grade": 95, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Bob Smith",        "student_id": "STU002", "course": "Mathematics",      "grade": 78, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Charlie Brown",    "student_id": "STU003", "course": "Physics",          "grade": 55, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Diana Prince",     "student_id": "STU004", "course": "English",          "grade": 48, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Eve Torres",       "student_id": "STU005", "course": "Biology",          "grade": 88, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Frank Castle",     "student_id": "STU006", "course": "Mathematics",      "grade": 52, "enrollment_date": "2024-09-01", "status": "inactive"},
    {"name": "Grace Hopper",     "student_id": "STU007", "course": "Computer Science", "grade": 91, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Hank Pym",         "student_id": "STU008", "course": "Physics",          "grade": 73, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Ivan Petrov",      "student_id": "STU009", "course": "Computer Science", "grade": 84, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Julia Child",      "student_id": "STU010", "course": "English",          "grade": 90, "enrollment_date": "2024-09-02", "status": "graduated"},
    {"name": "Karl Marx",        "student_id": "STU011", "course": "Mathematics",      "grade": 67, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Laura Palmer",     "student_id": "STU012", "course": "Biology",          "grade": 45, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Marco Polo",       "student_id": "STU013", "course": "Physics",          "grade": 82, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Nina Simone",      "student_id": "STU014", "course": "English",          "grade": 76, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Oscar Wilde",      "student_id": "STU015", "course": "Mathematics",      "grade": 93, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Pia Zadora",       "student_id": "STU016", "course": "Biology",          "grade": 71, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Quincy Adams",     "student_id": "STU017", "course": "Computer Science", "grade": 87, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Rosa Parks",       "student_id": "STU018", "course": "English",          "grade": 58, "enrollment_date": "2024-09-02", "status": "inactive"},
    {"name": "Sam Wilson",       "student_id": "STU019", "course": "Physics",          "grade": 98, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Tina Turner",      "student_id": "STU020", "course": "Mathematics",      "grade": 79, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Uma Thurman",      "student_id": "STU021", "course": "Computer Science", "grade": 62, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Vera Wang",        "student_id": "STU022", "course": "Biology",          "grade": 85, "enrollment_date": "2024-09-01", "status": "graduated"},
    {"name": "Walt Disney",      "student_id": "STU023", "course": "Physics",          "grade": 56, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Xena Warrior",     "student_id": "STU024", "course": "English",          "grade": 94, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Yusuf Islam",      "student_id": "STU025", "course": "Computer Science", "grade": 70, "enrollment_date": "2024-09-01", "status": "active"},
]

# insert_many() adds all documents to the collection in one call
result = students.insert_many(student_records)
print(f"Inserted {len(result.inserted_ids)} student records.")

### Step 3 — Query: Find Failing Students
The filter `{"grade": {"$lt": 60}}` matches failing students, and `{"_id": 0}` projects out the auto-generated `_id`.

In [ ]:
# Find all students with a grade below 60
# {"grade": {"$lt": 60}} means "grade less than 60"
# {"_id": 0} is a projection that hides MongoDB's auto-generated _id field
failing_students = students.find(
    {"grade": {"$lt": 60}},
    {"_id": 0}
)

print("--- Students Currently Failing (Grade < 60) ---")
for s in failing_students:
    print(f"{s['name']:<20} | {s['course']:<15} | Grade: {s['grade']} | Status: {s['status']}")

### Step 4 — Query: List CS Students Sorted by Grade
Chains a `course` filter with `.sort("grade", -1)` to list Computer Science students highest grade first.

In [ ]:
# Find all Computer Science students, sorted by grade descending (highest first)
cs_students = students.find(
    {"course": "Computer Science"},
    {"_id": 0}
).sort("grade", -1)  # -1 = descending order

print("--- Computer Science Students (Sorted by Grade) ---")
for s in cs_students:
    print(f"{s['name']:<20} | Grade: {s['grade']} | Status: {s['status']}")

### Step 5 — Query: Enrollment Count by Course (Aggregation)
`$group` buckets students by `course`, `$sum: 1` counts each group, and `$sort` orders the result.

In [ ]:
# Aggregation pipeline: group by course, then count students in each
pipeline = [
    {"$group": {"_id": "$course", "count": {"$sum": 1}}},
    {"$sort": {"count": -1, "_id": 1}}
]

course_counts = students.aggregate(pipeline)

print("--- Enrollment Count by Course ---")
for doc in course_counts:
    print(f"{doc['_id']:<20} | {doc['count']} students")

### Step 6 — Print Summary Report
Re-runs the Step 3–5 queries and prints them as one formatted administrator's report.

In [ ]:
# Re-run queries to collect data for the summary
failing = list(students.find({"grade": {"$lt": 60}}, {"_id": 0}))
top_cs = list(students.find({"course": "Computer Science"}, {"_id": 0}).sort("grade", -1))
enrollment = list(students.aggregate([
    {"$group": {"_id": "$course", "count": {"$sum": 1}}},
    {"$sort": {"count": -1, "_id": 1}}
]))

print("         STUDENT RECORDS SUMMARY REPORT")

print(f"\nTotal records in database: {students.count_documents({})}")
print(f"Students currently failing (grade < 60): {len(failing)}")

print("\n--- Failing Students ---")
for s in failing:
    print(f"  {s['name']} ({s['course']}) — Grade: {s['grade']}")

print(f"\n--- Top CS Student ---")
if top_cs:
    top = top_cs[0]
    print(f"  {top['name']} — Grade: {top['grade']}")

print("\n--- Course Enrollment ---")
for doc in enrollment:
    print(f"  {doc['_id']}: {doc['count']} students")
